# TTNS Chebyshev spectroscopy

One-site and two-site tangent Chebyshev propagation, and Burg spectral maximum entropy.

The two blocks reproduce the selected **CH3CN IR (D16)** and **298 K aggregate
thermal emission (D32)** comparisons. 
`MODE="replay"` rebuilds figures on CPU from archived arrays and timings.

## Main functions

| Function | Definition and purpose |
|---|---|
| `load_reference` / `prepare_reference` | Load checked states / run DMRG or purified thermal preparation |
| `tangent_chebyshev_1site` | Fixed one-site projector, matrix-free recurrence, doubled moments |
| `tangent_chebyshev_2site` | Complete one-site space plus both-end-discarded edge sectors |
| `estimate_tangent_bounds` | Separate matrix-free endpoint setup |
| `burg_maximum_entropy` / `extend_moments` | Covariance/Levinson maximum entropy and scalar continuation |
| `reconstruct_spectrum` | Gaussian quadrature or rational Lorentzian reconstruction |
| `cocheb` | Old cocheb code, variational compression, actual S/H, canonical reduction |
| `run_tangent` | Model-level tangent + Burg calculation, with `two_site=True/False` |
| `reproduce_ir` / `reproduce_emission` | The complete two comparison blocks below |



In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "ttns_spectra").exists():
    raise RuntimeError("Start this notebook from the extracted package root")
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("RENO_GPU", "0")
os.environ.setdefault("RENO_LOG_LEVEL", "30")

from IPython.display import display, Image, Markdown
from ttns_spectra import reproduce_ir, reproduce_emission, publication_settings

MODE = os.environ.get("TTNS_RUN_MODE", "fresh")
OUTPUT = ROOT / "outputs"
print("Execution mode:", MODE)
print("Slurm job:", os.environ.get("SLURM_JOB_ID", "local workstation"))
print("IR settings:", publication_settings("IR"))
print("Emission settings:", publication_settings("fast"))

## Block 1 — CH3CN infrared spectrum and runtime

Binary TTNS D16; sum of x/y/z. One-site TaChe: 250 projected actions / 500
computed moments. coCheb: 500 compressed vectors (499 recurrence updates).
One-site TDVP: dt=320 au, 175 steps, T=56000 au. Gaussian sigma=10 cm^-1.
The independent reference is the saved MPS D50 two-site TDVP curve.


In [ ]:
ir = reproduce_ir(mode=MODE, output=OUTPUT)
display(Image(filename=str(Path(ir["figure"]).with_suffix(".png"))))
display(Markdown("| Method | Response seconds | Published seconds | Relative spectrum difference |\n"
                 "|---|---:|---:|---:|\n" + "\n".join(
    f"| {r['method']} | {r['seconds']:.3f} | {r['published_seconds']:.3f} | {r['relative_difference_from_publication']:.3e} |"
    for r in ir["rows"])))

## Block 2 — Aggregate thermal emission and runtime

Open L10 aggregate, 298 K, J1=700 cm^-1, J2=87.5 cm^-1. Fast vibration:
1400 cm^-1, Huang–Rhys S=1, five local levels; binary TTNS D32. One-site
TaChe: 75 projected actions / 150 moments; coCheb: 150 vectors. One-site
TDVP: scaled dt=0.5, 60 steps, T=30. Lorentzian HWHM=0.1 scaled units
(140 cm^-1). The independent reference is the saved D64 chain two-site TDVP.


In [ ]:
emission = reproduce_emission(mode=MODE, output=OUTPUT)
display(Image(filename=str(Path(emission["figure"]).with_suffix(".png"))))
display(Markdown("| Method | Response seconds | Published seconds | Relative spectrum difference |\n"
                 "|---|---:|---:|---:|\n" + "\n".join(
    f"| {r['method']} | {r['seconds']:.3f} | {r['published_seconds']:.3f} | {r['relative_difference_from_publication']:.3e} |"
    for r in emission["rows"])))